In [ ]:
# ==========================================
# MODEL EVALUATION
# RideWise Customer Churn
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

In [ ]:
# ==========================================
# LOAD CORRECT MODELLING DATASET
# ==========================================

data_path = Path("../data/processed/rider_level_churn_dataset.csv")

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

df.head()

In [ ]:
# ==========================================
# DEFINE TARGET AND FEATURES
# Must match Notebook 05
# ==========================================

target = "is_churned"

drop_cols = [
    "is_churned",
    "user_id",
    "signup_date",
    "referred_by"
]

y = df[target]
X = df.drop(columns=drop_cols, errors="ignore")

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget distribution percentage:")
print(y.value_counts(normalize=True) * 100)

In [ ]:
# ==========================================
# TRAIN-TEST SPLIT
# Must match Notebook 05
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
# ==========================================
# LOAD BEST SAVED MODEL AND THRESHOLD
# ==========================================

model_dir = Path("../models")

best_model = joblib.load(model_dir / "churn_prediction_model.pkl")
best_threshold = joblib.load(model_dir / "churn_prediction_threshold.pkl")

best_threshold = round(float(best_threshold), 2)

print("Best model loaded successfully.")
print("Best threshold:", best_threshold)
print("Model type:", type(best_model))

In [ ]:
# ==========================================
# MAKE PREDICTIONS
# ==========================================

y_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= best_threshold).astype(int)

print("Predictions completed.")
print("Predicted Not Churn:", (y_pred == 0).sum())
print("Predicted Churn:", (y_pred == 1).sum())

In [ ]:
# ==========================================
# FINAL MODEL EVALUATION METRICS
# ==========================================

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_proba)
avg_precision = average_precision_score(y_test, y_proba)

evaluation_results = pd.DataFrame([
    {
        "Model": "Final Churn Model",
        "Threshold": best_threshold,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc,
        "Average Precision": avg_precision
    }
])

evaluation_results

In [ ]:
# ==========================================
# PRINT FINAL METRICS
# ==========================================

print("Final Model Evaluation")
print("----------------------------------------")
print(f"Threshold:          {best_threshold}")
print(f"Accuracy:           {accuracy:.4f}")
print(f"Precision:          {precision:.4f}")
print(f"Recall:             {recall:.4f}")
print(f"F1 Score:           {f1:.4f}")
print(f"ROC-AUC:            {roc_auc:.4f}")
print(f"Average Precision:  {avg_precision:.4f}")

In [ ]:
# ==========================================
# CLASSIFICATION REPORT
# ==========================================

print("Classification Report")
print("----------------------------------------")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# ==========================================
# CONFUSION MATRIX
# ==========================================

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Churn", "Churn"]
)

disp.plot()
plt.title("Confusion Matrix - Final Churn Model")
plt.show()

In [ ]:
# ==========================================
# ROC CURVE
# ==========================================

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("ROC Curve - Final Churn Model")
plt.grid(True)
plt.show()

In [ ]:
# ==========================================
# PRECISION-RECALL CURVE
# ==========================================

PrecisionRecallDisplay.from_predictions(y_test, y_proba)
plt.title("Precision-Recall Curve - Final Churn Model")
plt.grid(True)
plt.show()

In [ ]:
# ==========================================
# THRESHOLD EVALUATION
# ==========================================

thresholds = np.arange(0.20, 0.61, 0.05)

threshold_results = []

for threshold in thresholds:
    temp_pred = (y_proba >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(float(threshold), 2),
        "Accuracy": accuracy_score(y_test, temp_pred),
        "Precision": precision_score(y_test, temp_pred, zero_division=0),
        "Recall": recall_score(y_test, temp_pred, zero_division=0),
        "F1 Score": f1_score(y_test, temp_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.sort_values(by="F1 Score", ascending=False).reset_index(drop=True)

In [ ]:
# ==========================================
# PLOT F1 SCORE BY THRESHOLD
# ==========================================

plt.figure(figsize=(8, 6))
plt.plot(threshold_df["Threshold"], threshold_df["F1 Score"], marker="o")

plt.xlabel("Threshold")
plt.ylabel("F1 Score")
plt.title("F1 Score by Classification Threshold")
plt.grid(True)
plt.show()

In [ ]:
# ==========================================
# SAVE EVALUATION RESULTS
# ==========================================

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

evaluation_results.to_csv(
    reports_dir / "final_model_evaluation_results.csv",
    index=False
)

threshold_df.to_csv(
    reports_dir / "threshold_evaluation_results.csv",
    index=False
)

print("Evaluation results saved successfully.")

In [ ]:
# ==========================================
# SAVE FINAL PREDICTIONS
# ==========================================

predictions_df = X_test.copy()
predictions_df["actual_churn"] = y_test.values
predictions_df["predicted_churn"] = y_pred
predictions_df["churn_probability"] = y_proba
predictions_df["threshold"] = best_threshold

predictions_df.to_csv(
    reports_dir / "final_model_predictions.csv",
    index=False
)

print("Predictions saved successfully.")
predictions_df.head()